# Exercise 4.2: Open-Meteo weather for Frankfurt bike-sharing stations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/04_spatial_data_analysis/notebooks/exercise_4_2_openmeteo_bikesharing_frankfurt.ipynb)

This exercise reads Frankfurt bike-sharing station data from the course data loader, selects a short bike-sharing time window, fetches matching historical weather features from Open-Meteo, and visualizes the result on a time-slider map.

Open-Meteo returns hourly gridded weather. Each bike-sharing observation is matched to the weather hour at the same station coordinate.

In [ ]:
!pip -q install pandas folium requests branca

In [ ]:
from pathlib import Path
import json
import time
import zipfile
import urllib.request

import branca.colormap as cm
import folium
import pandas as pd
import requests
from folium.plugins import TimestampedGeoJson
from IPython.display import display

pd.set_option("display.max_columns", 100)

## 1. Load Frankfurt bike-sharing data

The original full European bike-sharing archive stays outside GitHub. This notebook reads the small Frankfurt-only filtered zip produced by the `data_loaders/frankfurt_bike_sharing` workflow.

In [ ]:
ZIP_NAME = "frankfurt_bike_sharing_full_filtered.zip"
SEAFILE_ZIP_URL = "https://seafile.rlp.net/seafhttp/f/81218a2c4f2c4ee28824/?op=view"

ROOT = Path.cwd()
if not (ROOT / "data_loaders" / "frankfurt_bike_sharing").exists():
    candidate = Path("/content/KI_Geodatenanalyse_SS26")
    if (candidate / "data_loaders" / "frankfurt_bike_sharing").exists():
        ROOT = candidate

candidate_paths = [
    ROOT / "data_loaders" / "frankfurt_bike_sharing" / "data" / ZIP_NAME,
    Path("/content") / ZIP_NAME,
    Path.cwd() / ZIP_NAME,
]


def seafile_download_candidates(url):
    candidates = [url]
    if "op=view" in url:
        candidates.append(url.replace("op=view", "op=download"))
    if "dl=1" not in url:
        sep = "&" if "?" in url else "?"
        candidates.append(f"{url}{sep}dl=1")
    return list(dict.fromkeys(candidates))


def download_zip(url, target):
    last_error = None
    for download_url in seafile_download_candidates(url):
        try:
            print("Trying:", download_url)
            urllib.request.urlretrieve(download_url, target)
            if zipfile.is_zipfile(target):
                return target
            last_error = RuntimeError("Downloaded file is not a valid zip.")
        except Exception as exc:
            last_error = exc
    if target.exists():
        target.unlink()
    raise RuntimeError(f"Could not download a valid zip: {last_error}")


zip_path = next((path for path in candidate_paths if path.exists() and zipfile.is_zipfile(path)), None)
if zip_path is None:
    zip_path = Path("/content") / ZIP_NAME if Path("/content").exists() else Path.cwd() / ZIP_NAME
    zip_path = download_zip(SEAFILE_ZIP_URL, zip_path)

extract_dir = Path("/content/frankfurt_bike_sharing_full") if Path("/content").exists() else zip_path.parent / "frankfurt_bike_sharing_full"
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

data_dir = extract_dir / "frankfurt_full"
print("Zip:", zip_path)
print("Data directory:", data_dir)

In [ ]:
stations = pd.read_csv(data_dir / "stations_frankfurt.csv")
station_status = pd.read_csv(data_dir / "station_status_frankfurt.csv")

stations["id"] = stations["id"].astype(str)
station_status["station_id"] = station_status["station_id"].astype(str)
station_status["datetime_utc"] = pd.to_datetime(station_status["time"], unit="s", utc=True)
station_status["datetime_berlin"] = station_status["datetime_utc"].dt.tz_convert("Europe/Berlin")

print(f"Stations: {len(stations):,}")
print(f"Station-status rows: {len(station_status):,}")
print(
    "Status time range:",
    station_status["datetime_berlin"].min(),
    "to",
    station_status["datetime_berlin"].max(),
)

display(stations.head())
display(station_status.head())

## 2. Select a short bike-sharing observation window

To keep Open-Meteo API usage small, the exercise uses a short time window. Within that window, all Frankfurt stations with bike-sharing observations are kept. For the bike-sharing table, each station contributes its first recorded status observation plus the following 20 observations when available.

In [ ]:
WINDOW_START = "2022-08-26 16:00"
WINDOW_HOURS = 24
OBSERVATIONS_AFTER_FIRST = 20
OBSERVATIONS_PER_STATION = OBSERVATIONS_AFTER_FIRST + 1

window_start = pd.Timestamp(WINDOW_START, tz="Europe/Berlin")
window_end = window_start + pd.Timedelta(hours=WINDOW_HOURS)

status_window = station_status[
    (station_status["datetime_berlin"] >= window_start)
    & (station_status["datetime_berlin"] < window_end)
].copy()

if status_window.empty:
    raise ValueError(f"No station-status observations found between {window_start} and {window_end}.")

selected_station_ids = sorted(status_window["station_id"].unique())
selected_stations = stations[stations["id"].isin(selected_station_ids)].copy()

selected_status = (
    status_window[status_window["station_id"].isin(selected_station_ids)]
    .sort_values(["station_id", "datetime_berlin"])
    .groupby("station_id", as_index=False)
    .head(OBSERVATIONS_PER_STATION)
    .merge(stations[["id", "name", "lat", "lon", "bike_racks"]], left_on="station_id", right_on="id", how="left")
    .drop(columns=["id"])
)
selected_status["weather_hour"] = selected_status["datetime_berlin"].dt.floor("h")
selected_status["weather_date"] = selected_status["weather_hour"].dt.strftime("%Y-%m-%d")

frame_hours = pd.date_range(
    window_start.floor("h"),
    (window_end - pd.Timedelta(seconds=1)).floor("h"),
    freq="1h",
    tz="Europe/Berlin",
)
frame_dates = pd.Series(frame_hours.strftime("%Y-%m-%d")).drop_duplicates().tolist()

print("Window:", window_start, "to", window_end)
print(f"Bike-sharing observations in window: {len(status_window):,}")
print(f"Stations represented: {len(selected_stations):,}")
print(f"Selected first+20 bike-sharing observations: {len(selected_status):,}")
print(f"Weather dates needed: {len(frame_dates):,}")
print(f"Station-date API coordinate pairs: {len(selected_stations) * len(frame_dates):,}")
display(selected_status.head())

## 3. Fetch Open-Meteo weather features

Open-Meteo's historical archive API supports multiple coordinates in one request. The short default window keeps request volume low while still fetching weather for every station represented in the window. Results are cached in `/content/openmeteo_frankfurt_cache` during the Colab session.

In [ ]:
OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FEATURES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
]
BATCH_SIZE = 40
REQUEST_SLEEP_SECONDS = 0.2
CACHE_DIR = Path("/content/openmeteo_frankfurt_cache") if Path("/content").exists() else Path("openmeteo_frankfurt_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def chunked(items, size):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def cache_path_for(date_text, station_ids):
    first = station_ids[0]
    last = station_ids[-1]
    return CACHE_DIR / f"openmeteo_{date_text}_{len(station_ids)}_{first}_{last}.json"


def fetch_openmeteo_batch(date_text, station_batch):
    station_ids = station_batch["station_id"].tolist()
    path = cache_path_for(date_text, station_ids)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    params = {
        "latitude": ",".join(station_batch["lat"].map(lambda value: f"{value:.6f}")),
        "longitude": ",".join(station_batch["lon"].map(lambda value: f"{value:.6f}")),
        "start_date": date_text,
        "end_date": date_text,
        "hourly": ",".join(WEATHER_FEATURES),
        "timezone": "Europe/Berlin",
        "wind_speed_unit": "ms",
        "precipitation_unit": "mm",
    }
    response = requests.get(OPEN_METEO_ARCHIVE_URL, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    path.write_text(json.dumps(payload), encoding="utf-8")
    time.sleep(REQUEST_SLEEP_SECONDS)
    return payload


def normalize_openmeteo_payload(payload):
    if isinstance(payload, list):
        return payload
    return [payload]


def weather_rows_from_payload(date_text, station_batch, payload):
    rows = []
    for station_row, location_payload in zip(station_batch.to_dict("records"), normalize_openmeteo_payload(payload)):
        hourly = location_payload.get("hourly", {})
        times = hourly.get("time", [])
        for index, time_text in enumerate(times):
            row = {
                "station_id": station_row["station_id"],
                "weather_hour": pd.Timestamp(time_text, tz="Europe/Berlin"),
                "weather_date": date_text,
                "openmeteo_latitude": location_payload.get("latitude"),
                "openmeteo_longitude": location_payload.get("longitude"),
            }
            for feature in WEATHER_FEATURES:
                values = hourly.get(feature, [])
                row[feature] = values[index] if index < len(values) else pd.NA
            rows.append(row)
    return rows


station_date_pairs = (
    selected_stations.rename(columns={"id": "station_id"})[["station_id", "name", "lat", "lon"]]
    .assign(key=1)
    .merge(pd.DataFrame({"weather_date": frame_dates, "key": 1}), on="key", how="inner")
    .drop(columns=["key"])
    .sort_values(["weather_date", "station_id"])
)

weather_rows = []
request_batches = 0
for date_text, date_stations in station_date_pairs.groupby("weather_date", sort=True):
    date_stations = date_stations.sort_values("station_id").reset_index(drop=True)
    for station_batch in chunked(date_stations, BATCH_SIZE):
        request_batches += 1
        payload = fetch_openmeteo_batch(date_text, station_batch)
        weather_rows.extend(weather_rows_from_payload(date_text, station_batch, payload))

weather = pd.DataFrame(weather_rows)
print(f"Open-Meteo request batches: {request_batches:,}")
print(f"Weather rows fetched: {len(weather):,}")
display(weather.head())

## 4. Join bike-sharing observations with weather

In [ ]:
bike_weather = selected_status.merge(
    weather,
    on=["station_id", "weather_hour", "weather_date"],
    how="left",
)

missing_weather = bike_weather[WEATHER_FEATURES].isna().all(axis=1).sum()
print(f"Bike-weather rows: {len(bike_weather):,}")
print(f"Rows without weather match: {missing_weather:,}")

display(
    bike_weather[
        [
            "station_id",
            "name",
            "datetime_berlin",
            "bikes_available_to_rent",
            "temperature_2m",
            "precipitation",
            "rain",
            "relative_humidity_2m",
            "cloud_cover",
            "wind_speed_10m",
        ]
    ].head()
)

## 5. Time-slider weather map

The map shows every station represented in the selected short window at every hourly slider step. Bike availability is forward-filled within each station so markers do not appear and disappear. Marker size is fixed; color represents temperature. Rain, precipitation, humidity, cloud cover, wind, and available bikes are shown in the popup.

In [ ]:
status_for_frames = (
    status_window.sort_values(["station_id", "datetime_berlin"])
    .merge(stations[["id", "name", "lat", "lon", "bike_racks"]], left_on="station_id", right_on="id", how="left")
    .drop(columns=["id"])
)
status_for_frames["weather_hour"] = status_for_frames["datetime_berlin"].dt.floor("h")

station_frames = (
    selected_stations.rename(columns={"id": "station_id"})[["station_id", "name", "lat", "lon", "bike_racks"]]
    .assign(key=1)
    .merge(pd.DataFrame({"weather_hour": frame_hours, "key": 1}), on="key", how="inner")
    .drop(columns=["key"])
    .sort_values(["station_id", "weather_hour"])
)
station_frames["weather_date"] = station_frames["weather_hour"].dt.strftime("%Y-%m-%d")

bike_hourly = (
    status_for_frames.sort_values(["station_id", "datetime_berlin"])
    .groupby(["station_id", "weather_hour"], as_index=False)
    .tail(1)[["station_id", "weather_hour", "datetime_berlin", "bikes_available_to_rent", "bikes", "free_racks"]]
)

map_data = station_frames.merge(bike_hourly, on=["station_id", "weather_hour"], how="left")
map_data[["bikes_available_to_rent", "bikes", "free_racks"]] = (
    map_data.groupby("station_id")[["bikes_available_to_rent", "bikes", "free_racks"]]
    .ffill()
)
map_data = map_data.merge(
    weather,
    on=["station_id", "weather_hour", "weather_date"],
    how="left",
)
map_data["temperature_2m"] = pd.to_numeric(map_data["temperature_2m"], errors="coerce")
map_data = map_data.dropna(subset=["temperature_2m"]).copy()
map_data["precipitation"] = pd.to_numeric(map_data["precipitation"], errors="coerce").fillna(0)
map_data["bikes_available_to_rent"] = pd.to_numeric(map_data["bikes_available_to_rent"], errors="coerce")

temp_min = float(map_data["temperature_2m"].min())
temp_max = float(map_data["temperature_2m"].max())
temp_colormap = cm.LinearColormap(
    colors=["#2c7bb6", "#ffffbf", "#d7191c"],
    vmin=temp_min,
    vmax=temp_max,
    caption="Temperature at station coordinate (deg C)",
)

features = []
for _, row in map_data.iterrows():
    temperature = row["temperature_2m"]
    precipitation = row["precipitation"]
    available_bikes = row["bikes_available_to_rent"]
    popup = (
        f"<b>{row['name']}</b><br>"
        f"Station id: {row['station_id']}<br>"
        f"Map hour: {row['weather_hour']}<br>"
        f"Available bikes: {available_bikes if pd.notna(available_bikes) else 'n/a'}<br>"
        f"Temperature: {temperature:.1f} deg C<br>"
        f"Precipitation: {precipitation:.2f} mm<br>"
        f"Rain: {row['rain']} mm<br>"
        f"Humidity: {row['relative_humidity_2m']} %<br>"
        f"Cloud cover: {row['cloud_cover']} %<br>"
        f"Wind speed: {row['wind_speed_10m']} m/s"
    )
    features.append(
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
            "properties": {
                "time": row["weather_hour"].isoformat(),
                "popup": popup,
                "tooltip": f"{row['name']}: {temperature:.1f} deg C, {precipitation:.2f} mm",
                "icon": "circle",
                "iconstyle": {
                    "fillColor": temp_colormap(temperature),
                    "fillOpacity": 0.82,
                    "stroke": "true",
                    "color": "#333333",
                    "weight": 1,
                    "radius": 6,
                },
                "style": {"color": "#333333"},
            },
        }
    )

weather_map = folium.Map(
    location=[map_data["lat"].mean(), map_data["lon"].mean()],
    zoom_start=12,
    tiles="cartodbpositron",
)
TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="PT1H",
    duration="PT1H",
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=2,
    loop_button=True,
    date_options="YYYY-MM-DD HH:mm",
    time_slider_drag_update=True,
).add_to(weather_map)
temp_colormap.add_to(weather_map)
weather_map

## 6. Questions

1. Which stations first appear during rainy hours?
2. Does temperature or rain vary much across Frankfurt station coordinates, or mainly over time?
3. Why is hourly weather matched to minute-level bike-sharing observations? What uncertainty does that introduce?
4. How would the analysis change if we joined full daily weather instead of hourly weather?